# Lab 3: Morphology

Compute the similarity of the *SMTeuroparl* sentence pairs (test set) using lemmas and the Jaccard distance. Both the nltk and spaCy lemmatizers
 are used in this exercise for comparison.

In [1]:
from pathlib import Path

# The notebook runs from labs/, so the workspace root is the parent folder.
ROOT = Path.cwd().parent

DATA_DIR = ROOT / "data" / "test_gold"
NLTK_DIR = ROOT / "models" / "nltk_data"

# Notebook libs
import pandas as pd
import nltk
from nltk.metrics.distance import jaccard_distance
from scipy.stats import pearsonr
import spacy

# NLTK packages needed by this notebook
nltk.data.path.insert(0, str(NLTK_DIR))
nltk.download("punkt_tab", download_dir=NLTK_DIR)
nltk.download("averaged_perceptron_tagger_eng", download_dir=NLTK_DIR)
nltk.download("wordnet", download_dir=NLTK_DIR)
nltk.download("omw-1.4", download_dir=NLTK_DIR)

# SpaCy model needed by this notebook
if not spacy.util.is_package("en_core_web_sm"):
    spacy.cli.download("en_core_web_sm")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Load SpaCy model
nlp = spacy.load("en_core_web_sm")

# Load sentence pairs and gold-standard similarity scores
dt = pd.read_csv(DATA_DIR / 'STS.input.SMTeuroparl.txt', sep='\t', header=None)
dt['gs'] = pd.read_csv(DATA_DIR / 'STS.gs.SMTeuroparl.txt', sep='\t', header=None)

In [3]:
# Store the Jaccard similarities calculated using words and lemmas with spaCy
jaccards = []
jaccards_lemma = []

for i in range(len(dt)):
    # Get the two sentences from the current sentence pair.
    s1 = dt.iloc[i, 0]
    s2 = dt.iloc[i, 1]

    # Process the sentences with spaCy to obtain their lemmas.
    doc1 = nlp(s1)
    doc2 = nlp(s2)

    # Tokenize the sentences and represent the words as sets.
    words1 = set(nltk.word_tokenize(s1))
    words2 = set(nltk.word_tokenize(s2))

    # Extract the lemmas and represent them as sets.
    lemma1 = {token.lemma_ for token in doc1}
    lemma2 = {token.lemma_ for token in doc2}

    # Calculate Jaccard distance using words and convert it to similarity.
    j_distance = jaccard_distance(words1, words2)
    j_similarity = 1 - j_distance
    jaccards.append(j_similarity)

    # Calculate Jaccard distance using lemmas and convert it to similarity.
    j_distance_lemma = jaccard_distance(lemma1, lemma2)
    j_similarity_lemma = 1 - j_distance_lemma
    jaccards_lemma.append(j_similarity_lemma)

# Add both similarity scores to the dataset for comparison.
dt['jaccard'] = jaccards
dt['jaccard_lemma'] = jaccards_lemma

# Compute the pearson correlation
pearson = pearsonr(dt['gs'], dt['jaccard'])[0]
pearson_spaCy_lemma = pearsonr(dt['gs'], dt['jaccard_lemma'])[0]

In [4]:
# Function to print pairs to help with the justification
def show_pair(i):
    s1, s2 = dt.iloc[i, 0], dt.iloc[i, 1]
    doc1, doc2 = nlp(s1), nlp(s2)

    print(f"Pair {i}  (gold {dt['gs'][i]})")
    print(f"  S1: {s1}")
    print(f"  S2: {s2}")

    representations = [
        ("Words", set(nltk.word_tokenize(s1)), set(nltk.word_tokenize(s2)), dt['jaccard'][i]),
        ("Lemmas", {t.lemma_ for t in doc1}, {t.lemma_ for t in doc2}, dt['jaccard_lemma'][i]),
    ]
    for name, set1, set2, similarity in representations:
        print(f"  {name} (Jaccard {similarity:.3f})")
        print(f"    shared:  {sorted(set1 & set2)}")
        print(f"    only S1: {sorted(set1 - set2)}")
        print(f"    only S2: {sorted(set2 - set1)}")
    print()

In [5]:
# Print the pearson correlation 
print(f"Pearson correlation: {pearson:.4f}")
print(f"Pearson correlation using SpaCy lemma: {pearson_spaCy_lemma:.4f}")

# Count the pairs whose similarity increases / decreases / stays equal with spaCy lemmas.
diff = dt['jaccard_lemma'] - dt['jaccard']

print(f"Total pairs:   {len(dt)}")
print(f"Lemmas higher: {(diff > 0).sum()}")
print(f"Lemmas lower:  {(diff < 0).sum()}")
print(f"Equal:         {(diff == 0).sum()}")

Pearson correlation: 0.4507
Pearson correlation using SpaCy lemma: 0.4784
Total pairs:   459
Lemmas higher: 229
Lemmas lower:  23
Equal:         207


In [6]:
# Find examples: the pairs whose similarity changes the most with lemmas, top 3.
print("=== Largest increases with lemmas ===\n")
for i in diff.nlargest(3).index:
    show_pair(i)

print("=== Largest decreases with lemmas ===\n")
for i in diff.nsmallest(3).index:
    show_pair(i)

=== Largest increases with lemmas ===

Pair 12  (gold 4.75)
  S1: (Parliament adopted the legislative resolution)
  S2: (The Parliament adopts legislative resolution)
  Words (Jaccard 0.556)
    shared:  ['(', ')', 'Parliament', 'legislative', 'resolution']
    only S1: ['adopted', 'the']
    only S2: ['The', 'adopts']
  Lemmas (Jaccard 1.000)
    shared:  ['(', ')', 'Parliament', 'adopt', 'legislative', 'resolution', 'the']
    only S1: []
    only S2: []

Pair 350  (gold 5.0)
  S1: Then perhaps we could have avoided a catastrophe.
  S2: We could perhaps then have avoided a catastrophe.
  Words (Jaccard 0.636)
    shared:  ['.', 'a', 'avoided', 'catastrophe', 'could', 'have', 'perhaps']
    only S1: ['Then', 'we']
    only S2: ['We', 'then']
  Lemmas (Jaccard 1.000)
    shared:  ['.', 'a', 'avoid', 'catastrophe', 'could', 'have', 'perhaps', 'then', 'we']
    only S1: []
    only S2: []

Pair 49  (gold 5.0)
  S1: (Parliament adopted the legislative resolution)
  S2: (The Parliament ado

In [7]:
# Check for repeated sentence pairs in the dataset.
duplicated = dt.duplicated(subset=[0, 1])
print(f"Repeated pairs: {duplicated.sum()} of {len(dt)} ({(~duplicated).sum()} distinct)")

# Check for repeated pairs in the case that got worse with lemmas.
worse = dt[diff < 0]
worse_duplicated = worse.duplicated(subset=[0, 1])
print(f"Repeated pairs in the case that lemma got worse: {worse_duplicated.sum()} of {len(worse_duplicated)} ({(~worse_duplicated).sum()} distinct)")

Repeated pairs: 142 of 459 (317 distinct)
Repeated pairs in the case that lemma got worse: 11 of 23 (12 distinct)


In [8]:
# Pair 260 (largest decrease): Print the POS tags to see why
show_pair(260)
for s in [dt.iloc[260, 0], dt.iloc[260, 1]]:
    print([(t.text, t.pos_, t.lemma_) for t in nlp(s)])

# Pair 9 (second largest decrease):
show_pair(9)
for s in [dt.iloc[9, 0], dt.iloc[9, 1]]:
    print([(t.text, t.pos_, t.lemma_) for t in nlp(s)])


Pair 260  (gold 4.75)
  S1: Van Orden Report (A5-0241/2000)
  S2: Report Van Orden (A5-0241/2000)
  Words (Jaccard 1.000)
    shared:  ['(', ')', 'A5-0241/2000', 'Orden', 'Report', 'Van']
    only S1: []
    only S2: []
  Lemmas (Jaccard 0.778)
    shared:  ['(', ')', '-', '0241/2000', 'A5', 'Orden', 'Van']
    only S1: ['Report']
    only S2: ['report']

[('Van', 'PROPN', 'Van'), ('Orden', 'PROPN', 'Orden'), ('Report', 'PROPN', 'Report'), ('(', 'PUNCT', '('), ('A5', 'PROPN', 'A5'), ('-', 'PUNCT', '-'), ('0241/2000', 'NUM', '0241/2000'), (')', 'PUNCT', ')')]
[('Report', 'NOUN', 'report'), ('Van', 'PROPN', 'Van'), ('Orden', 'PROPN', 'Orden'), ('(', 'PUNCT', '('), ('A5', 'PROPN', 'A5'), ('-', 'PUNCT', '-'), ('0241/2000', 'NUM', '0241/2000'), (')', 'PUNCT', ')')]
Pair 9  (gold 5.0)
  S1: The vote will take place today at 5.30 p.m.
  S2: The vote will take place at 17h30.
  Words (Jaccard 0.636)
    shared:  ['.', 'The', 'at', 'place', 'take', 'vote', 'will']
    only S1: ['5.30', 'p.m', '

In [9]:
# Compare with the NLTK lemmatizer
wnl = nltk.stem.WordNetLemmatizer()

def lemmatize(p):
    d = {'NN': 'n', 'NNS': 'n',
         'JJ': 'a', 'JJR': 'a', 'JJS': 'a',
         'VB': 'v', 'VBD': 'v', 'VBG': 'v', 'VBN': 'v', 'VBP': 'v', 'VBZ': 'v',
         'RB': 'r', 'RBR': 'r', 'RBS': 'r'}
    if p[1] in d:
        return wnl.lemmatize(p[0], pos=d[p[1]])
    return p[0]

def nltk_lemmas(s):
    return {lemmatize(pair) for pair in nltk.pos_tag(nltk.word_tokenize(s))}

# To compute the jaccard distance using nltk lemma function 
dt['jaccard_nltk_lemma'] = [
    1 - jaccard_distance(nltk_lemmas(s1), nltk_lemmas(s2))
    for s1, s2 in zip(dt[0], dt[1])
]
pearson_nltk_lemma = pearsonr(dt['gs'], dt['jaccard_nltk_lemma'])[0]

print(f"Pearson correlation using words:        {pearson:.4f}")
print(f"Pearson correlation using NLTK lemmas:  {pearson_nltk_lemma:.4f}")
print(f"Pearson correlation using spaCy lemmas: {pearson_spaCy_lemma:.4f}")

diff_nltk = dt['jaccard_nltk_lemma'] - dt['jaccard']
print(f"\nNLTK lemmas higher: {(diff_nltk > 0).sum()}, lower: {(diff_nltk < 0).sum()}, equal: {(diff_nltk == 0).sum()}")

Pearson correlation using words:        0.4507
Pearson correlation using NLTK lemmas:  0.4494
Pearson correlation using spaCy lemmas: 0.4784

NLTK lemmas higher: 97, lower: 12, equal: 350


In [10]:
# To compare pairs 12 and 350 (largest increases with spaCy) with NLTK POS tags and lemmas.
for i in [12, 350]:
    print(f"Pair {i}  (NLTK lemmas Jaccard {dt['jaccard_nltk_lemma'][i]:.3f}, spaCy lemmas Jaccard {dt['jaccard_lemma'][i]:.3f})")
    for s in [dt.iloc[i, 0], dt.iloc[i, 1]]:
        print([(word, tag, lemmatize((word, tag))) for word, tag in nltk.pos_tag(nltk.word_tokenize(s))])
    print()

Pair 12  (NLTK lemmas Jaccard 0.556, spaCy lemmas Jaccard 1.000)
[('(', '(', '('), ('Parliament', 'NNP', 'Parliament'), ('adopted', 'VBD', 'adopt'), ('the', 'DT', 'the'), ('legislative', 'JJ', 'legislative'), ('resolution', 'NN', 'resolution'), (')', ')', ')')]
[('(', '(', '('), ('The', 'DT', 'The'), ('Parliament', 'NNP', 'Parliament'), ('adopts', 'NNS', 'adopts'), ('legislative', 'JJ', 'legislative'), ('resolution', 'NN', 'resolution'), (')', ')', ')')]

Pair 350  (NLTK lemmas Jaccard 0.636, spaCy lemmas Jaccard 1.000)
[('Then', 'RB', 'Then'), ('perhaps', 'RB', 'perhaps'), ('we', 'PRP', 'we'), ('could', 'MD', 'could'), ('have', 'VB', 'have'), ('avoided', 'VBN', 'avoid'), ('a', 'DT', 'a'), ('catastrophe', 'NN', 'catastrophe'), ('.', '.', '.')]
[('We', 'PRP', 'We'), ('could', 'MD', 'could'), ('perhaps', 'RB', 'perhaps'), ('then', 'RB', 'then'), ('have', 'VB', 'have'), ('avoided', 'VBN', 'avoid'), ('a', 'DT', 'a'), ('catastrophe', 'NN', 'catastrophe'), ('.', '.', '.')]



In [11]:
# Separate the effects of the tokenizer, lowercasing and lemmatization
# by computing the Jaccard similarity on different token representations.
docs = [(nlp(s1), nlp(s2)) for s1, s2 in zip(dt[0], dt[1])]

variants = {
    'spaCy tokens': lambda doc: {t.text for t in doc},
    'spaCy tokens, lowercased': lambda doc: {t.text.lower() for t in doc},
    'spaCy lemmas': lambda doc: {t.lemma_ for t in doc},
    'spaCy lemmas, lowercased': lambda doc: {t.lemma_.lower() for t in doc},
}

pearsons = {'NLTK words': pearson}
for name, extract in variants.items():
    sims = [1 - jaccard_distance(extract(d1), extract(d2)) for d1, d2 in docs]
    pearsons[name] = pearsonr(dt['gs'], sims)[0]

for name, p in pearsons.items():
    print(f"{name:<26} {p:.4f}")

# Gain of each step, from NLTK words to lowercased spaCy lemmas.
print(f"\nTokenizer (NLTK words -> spaCy tokens):          {pearsons['spaCy tokens'] - pearsons['NLTK words']:+.4f}")
print(f"Lowercasing (spaCy tokens -> lowercased):        {pearsons['spaCy tokens, lowercased'] - pearsons['spaCy tokens']:+.4f}")
print(f"Lemmatization (lowercased tokens -> lemmas):     {pearsons['spaCy lemmas, lowercased'] - pearsons['spaCy tokens, lowercased']:+.4f}")

NLTK words                 0.4507
spaCy tokens               0.4606
spaCy tokens, lowercased   0.4726
spaCy lemmas               0.4784
spaCy lemmas, lowercased   0.4811

Tokenizer (NLTK words -> spaCy tokens):          +0.0099
Lowercasing (spaCy tokens -> lowercased):        +0.0120
Lemmatization (lowercased tokens -> lemmas):     +0.0084


In [12]:
# Most pairs are near-paraphrases, and lemmas raise the similarities overall.
print(f"Mean gold score:            {dt['gs'].mean():.2f} (scale 0-5)")
print(f"Mean similarity with words:  {dt['jaccard'].mean():.2f}")
print(f"Mean similarity with lemmas: {dt['jaccard_lemma'].mean():.2f}")

# Lemmatization removes the tense: these two sentences describe different events.
s1, s2 = "The vote took place yesterday.", "The vote will take place tomorrow."
words1, words2 = set(nltk.word_tokenize(s1)), set(nltk.word_tokenize(s2))
lemma1, lemma2 = {t.lemma_ for t in nlp(s1)}, {t.lemma_ for t in nlp(s2)}
print(f"\nS1: {s1}\nS2: {s2}")
print(f"  Words  (Jaccard {1 - jaccard_distance(words1, words2):.3f}) shared: {sorted(words1 & words2)}")
print(f"  Lemmas (Jaccard {1 - jaccard_distance(lemma1, lemma2):.3f}) shared: {sorted(lemma1 & lemma2)}")

Mean gold score:            4.55 (scale 0-5)
Mean similarity with words:  0.50
Mean similarity with lemmas: 0.55

S1: The vote took place yesterday.
S2: The vote will take place tomorrow.
  Words  (Jaccard 0.444) shared: ['.', 'The', 'place', 'vote']
  Lemmas (Jaccard 0.625) shared: ['.', 'place', 'take', 'the', 'vote']


## Questions

### 1. Which is better: words or lemmas?

**Lemmas are better on this dataset, when using spaCy.** The Pearson correlation with the gold standard rises from **0.4507** (words) to **0.4784** (spaCy lemmas). For 229 of the 459 pairs the lemma similarity is higher than the word similarity, it is lower for only 23, and equal for the rest.

The reason is that lemmatization maps inflected forms to the same item, so paraphrases that differ only in morphology now share tokens. 

For example:

- **Pair 12** (with the largest increase of similarity score): *"(Parliament **adopted** the legislative resolution)"* vs *"(The Parliament **adopts** legislative resolution)"* (gold 4.75): words 0.556 → lemmas 1.0, because *adopted* and *adopts* both become *adopt*, and *The*/*the* are merged since the spaCy lemmas are lowercased except proper nouns.

**However, the result depends on the lemmatizer.** With the NLTK lemmatizer from the lab slides (POS tagger + WordNet lemmatizer), the Pearson correlation is **0.4494**, slightly lower than words, and only 97 pairs improve. 

For example: 

- **Pair 12**, the NLTK tagger tags *adopts* as a plural noun (NNS), so it stays *adopts*, and NLTK lemmas are not lowercased, so *The*/*the* are still different. The similarity stays at 0.556, the same as with words.

**The gain of spaCy lemmas is not due only to lemmatization.** Computing the similarity on different spaCy token representations separates the contributions:

| Representation | Pearson |
|---|---|
| NLTK words | 0.4507 |
| spaCy tokens | 0.4606 |
| spaCy tokens, lowercased | 0.4726 |
| spaCy lemmas | 0.4784 |
| spaCy lemmas, lowercased | 0.4811 |

- **Tokenizer** (NLTK words → spaCy tokens): +0.0099
- **Lowercasing** (spaCy tokens → lowercased): +0.0120
- **Lemmatization** (lowercased tokens → lowercased lemmas): +0.0084

So the tokenizer and lowercasing together bring more than the lemmatization itself. For example, in **pair 350** *"**Then** perhaps **we** could…"* vs *"**We** could perhaps **then**…"* the lemmas only reach 1.0 because *Then*/*then* and *We*/*we* are merged. The pure lemmatization effect is smaller, but still positive, so lemmas are better, although the margin is modest.

### 2. Do you think that could perform better for any pair of texts?

**No.** Lemmas are better on average, but not for every pair:

- **Lemmatization errors (pair 260, largest decrease).** Lemmas depend on the POS tagger, which depends on context. In *"Van Orden **Report** (A5-0241/2000)"* vs *"**Report** Van Orden (A5-0241/2000)"* (gold 4.75), *Report* is tagged as a proper noun in the first sentence (lemma *Report*) and as a common noun in the second (lemma *report*) when using the spaCy lemmas. The similarity drops from 1.0 to 0.78.
- **Tokenization differences (pair 9).** In *"…today at 5.30 **p.m.**"* vs *"…at 17h30."* (gold 5.0), spaCy keeps *p.m.* as one token, so the final punctuation period *"."* is no longer shared and the similarity drops from 0.636 to 0.545.
- **Loss of meaning.** Lemmatization removes information carried by inflection, such as tense and number. *"The vote **took** place yesterday."* vs *"The vote **will take** place tomorrow."* describe different events, but their similarity rises from 0.444 (words) to 0.625 (lemmas), because *took* and *take* both become *take*. In SMTeuroparl most pairs are near-paraphrases (mean gold score 4.55 out of 5), so merging forms usually helps, but in a dataset where these differences matter, lemmas could hurt.
- **Higher similarities overall.** Lemmas raise the mean similarity from 0.50 (words) to 0.55, which also makes unrelated pairs look more alike.

The dataset contains many repeated pairs. The 23 pairs that got worse turn out to be only 12 distinct pairs, so the cases where lemmas hurt are even fewer.

In addition, both approaches ignore synonyms (*5.30 p.m.* vs *17h30* pair), word order and sentence structure, so neither words nor lemmas alone can capture semantic similarity for all pairs.